# AwareLiquid-Tiny — TinyStories pretraining on Google Colab

A ~49M MT-LNN (d_model=416, 6 layers) trained on TinyStories (gpt2 BPE) for a **coherent** demo.

**Why Colab + Google Drive:** Colab free sessions disconnect (idle/timeout ~12h). All data and checkpoints live on **Google Drive** so any re-run **auto-resumes** from the newest checkpoint. The repo is cloned fresh from GitHub `main` each run, so the **double-shift label fix** (commit `b027bbf`) is always picked up.

**How to use:**
1. Runtime → Change runtime type → **GPU** (T4 is fine).
2. Run all cells top to bottom.
3. If it disconnects, just **Run all again** — it resumes from Drive.
4. When done, the file `serve.pt` (in your Drive `awareliquid_tiny/checkpoints/`) is the drop-in for the demo — download it and send it over.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'No GPU! Runtime -> Change runtime type -> Hardware accelerator -> GPU, then re-run.'
)
dev = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f'torch {torch.__version__} | device={dev} | capability=sm_{cap[0]}{cap[1]}')
# T4 = sm_75, supported by stock Colab torch (no Pascal reinstall needed).

## 2. Mount Google Drive (persistence + resume)

Everything heavy (tokenised `*.bin`, checkpoints) is written under `MyDrive/awareliquid_tiny/` so it survives disconnects.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/awareliquid_tiny'
DATA_DIR   = os.path.join(DRIVE_ROOT, 'data')
CKPT_DIR   = os.path.join(DRIVE_ROOT, 'checkpoints')
METRICS    = os.path.join(DRIVE_ROOT, 'metrics.jsonl')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 3. Clone the M1 repo (fresh from GitHub main → always has the label fix)

In [ ]:
import subprocess, sys
REPO = 'https://github.com/everest-an/M1.git'
DIR  = '/content/M1'
if not os.path.exists(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'pull', '--ff-only'], check=False)
os.chdir(DIR)
sys.path.insert(0, DIR)
print('repo HEAD:')
subprocess.run(['git', '-C', DIR, 'log', '--oneline', '-1'])

## 4. Install dependencies

Pin to Colab's existing torch so we never disturb the working CUDA build (T4 is supported out of the box — unlike Kaggle's Pascal P100).

In [ ]:
torch_public = torch.__version__.split('+')[0]
with open('/content/pip-constraints.txt', 'w') as f:
    f.write(f'torch=={torch_public}\n')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-c', '/content/pip-constraints.txt',
     'datasets', 'transformers', 'tokenizers', 'tqdm', 'einops'],
    check=True,
)
print('deps installed (torch pinned to', torch_public, ')')

## 5. Tokenise TinyStories → Drive (skipped if already done)

~472M train tokens / ~4.7M val tokens (gpt2 BPE). Done once; cached on Drive.

In [ ]:
if not os.path.exists(os.path.join(DATA_DIR, 'meta.json')):
    print('[data] tokenising roneneldan/TinyStories (gpt2) -> data/*.bin ...')
    # Run in a SUBPROCESS so a fresh interpreter does the heavy import. We pass
    # config=None explicitly because prepare_data's CLI default --config is
    # WikiText's and would error for TinyStories (which has no named config).
    tok = (
        'import prepare_data;from types import SimpleNamespace;'
        'prepare_data.main(SimpleNamespace(dataset="roneneldan/TinyStories",'
        'config=None,tokenizer="gpt2",out_dir=%r))' % DATA_DIR
    )
    subprocess.run([sys.executable, '-c', tok], check=True, cwd=DIR)
else:
    print('[data] reusing existing tokenised data/ on Drive')
import json
print(json.load(open(os.path.join(DATA_DIR, 'meta.json'))))

## 6. Resume detection (newest checkpoint on Drive)

In [ ]:
cands = [os.path.join(CKPT_DIR, n) for n in ('last.pt', 'final.pt')]
cands += [os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR) if f.startswith('ckpt_') and f.endswith('.pt')]
existing = [c for c in cands if os.path.exists(c)]
resume_args = []
if existing:
    resume_ckpt = max(existing, key=os.path.getmtime)
    resume_args = ['--resume', resume_ckpt]
    print('[resume] resuming from', resume_ckpt)
else:
    print('[resume] no checkpoint found — fresh start')

## 7. Train

Same config as the Kaggle kernel: ~49M params, v2 signature modules (competitive GWT + predictive world model) ON. Checkpoints save directly to Drive every 1000 steps so a disconnect loses at most ~1000 steps. Tunable via the env vars below.

In [ ]:
STEPS      = int(os.environ.get('T_STEPS', '20000'))
BATCH      = int(os.environ.get('T_BATCH', '12'))      # T4 15GB ~ P100 16GB
GRAD_ACCUM = int(os.environ.get('T_GRAD_ACCUM', '4'))  # global batch 48
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

cmd = [
    sys.executable, 'train.py',
    '--data_dir', DATA_DIR, '--ckpt_dir', CKPT_DIR,
    '--metrics_jsonl', METRICS, '--metrics_every', '50',
    '--d_model', '416', '--n_layers', '6', '--n_heads', '13', '--n_kv_heads', '1',
    '--gwtb_n_heads', '4', '--seq_len', '512',
    '--batch', str(BATCH), '--grad_accum', str(GRAD_ACCUM),
    '--lr', '6e-4', '--warmup_steps', '200', '--steps', str(STEPS),
    '--log_every', '100', '--eval_every', '500', '--eval_batches', '40',
    '--save_every', '1000',
    '--competitive_gwtb', '--n_bids', '3',
    '--world_model', '--world_model_weight', '0.01', '--world_model_grad_clip', '1.0',
] + resume_args
print('[train]', ' '.join(cmd), '\n')
subprocess.run(cmd, check=True)

## 8. Write slim, server-loadable `serve.pt`

Drops the optimizer state (~1/3 the size). This is the drop-in for `serve/server.py` and the demo. **Download this file** and send it over.

In [ ]:
import shutil
final = os.path.join(CKPT_DIR, 'final.pt')
last  = os.path.join(CKPT_DIR, 'last.pt')
src = final if os.path.exists(final) else (last if os.path.exists(last) else None)
assert src, 'no final.pt/last.pt found — did training finish?'
if src == final:
    shutil.copyfile(final, last)
# torch>=2.6 defaults weights_only=True, which can't unpickle the dataclass
# config; force a full unpickle.
try:
    ck = torch.load(src, map_location='cpu', weights_only=False)
except TypeError:
    ck = torch.load(src, map_location='cpu')
slim = {'config': ck['config'], 'model_state': ck['model_state'],
        'step': ck.get('step'), 'loss': ck.get('loss')}
serve_path = os.path.join(CKPT_DIR, 'serve.pt')
torch.save(slim, serve_path)
print(f'[serve] wrote {serve_path} ({os.path.getsize(serve_path)/1e6:.0f} MB)  step={slim["step"]} loss={slim["loss"]}')

# Optional: trigger a browser download straight from Colab.
try:
    from google.colab import files
    files.download(serve_path)
except Exception as e:
    print('(auto-download skipped — grab serve.pt from Drive manually):', e)